← [Overview](00_overview.ipynb)

# Rescaling and denormalisation

Clustering and [representation](06_representation.ipynb) both work on the *normalised* period
matrix. Two final steps turn the chosen representatives back into a usable series:

1. **Rescaling** — a non-mean representative can let the occurrence-weighted totals drift
   away from the original; tsam restores them (`preserve_column_means`).
2. **Denormalisation** — invert the min-max scaling so every value is back in physical units.

Both are computed live below on the tiny six-day set.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig

pio.renderers.default = "notebook_connected"

# Raw tiny series (for the column ranges) and the preprocessed period matrix D
# (normalised + unstacked, attribute-major) from 01_preprocessing.
tiny = pd.read_csv("../tiny.csv", index_col=0, parse_dates=True)
col_min, col_max = tiny.min(), tiny.max()
normalized = (tiny - col_min) / (col_max - col_min)

D = pd.read_csv("../tiny_periods.csv", header=[0, 1], index_col=0)
D_arr = D.values
N_PERIODS, N_ATTRS, N_TIMESTEPS = 6, 2, 4


# Reuse the k=3 Lloyd assignments + centroids from the partitional notebook.
def euclidean_dist(a, b):
    return float(np.sqrt(np.sum((a - b) ** 2)))


k = 3
centers = D_arr[[0, 2, 4]].copy().astype(float)
for _ in range(10):
    assignments = np.array(
        [np.argmin([euclidean_dist(D_arr[p], centers[kk]) for kk in range(k)])
         for p in range(N_PERIODS)]
    )
    new_centers = np.array(
        [D_arr[assignments == kk].mean(axis=0) if (assignments == kk).any()
         else centers[kk]
         for kk in range(k)]
    )
    if np.allclose(centers, new_centers):
        break
    centers = new_centers
centroid_0 = D_arr[assignments == 0].mean(axis=0)

# Real dataset (for the with/without-rescaling demonstration).
raw = pd.read_csv("../testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]
print("tiny:", tiny.shape, "  real:", data.shape, "  Lloyd assignments:", assignments)

tiny: (24, 2)   real: (1008, 4)   Lloyd assignments: [0 0 1 1 2 2]


---

## 1  Rescaling — why means can drift and how to correct them

After clustering, the occurrence-weighted sum of representatives may differ from
the original total. tsam corrects this with a multiplicative rescaling factor
(Hoffmann §3.2.2.3):

$$
c^*_{k,a,t} = c_{k,a,t} \cdot
\frac{\sum_{p=1}^{N_p}\sum_{t=1}^{N_t} x_{p,a,t}}
{\sum_{k=1}^{N_k} \left(|\mathbb{C}_k| \sum_{t=1}^{N_t} c_{k,a,t}\right)}
\quad \forall\; k, a, t
$$

The numerator is the total original sum; the denominator is what the
occurrence-weighted representatives sum to before rescaling. If they match,
the factor is 1. Enabled by default (`preserve_column_means=True`).

**TSAM configuration for rescaling:**

In [2]:
# Rescaling is controlled by preserve_column_means (top-level aggregate param).
# It is NOT part of ClusterConfig — it applies after all clustering is done.

# Default (preserve_column_means=True): rescale so weighted mean of
# representatives matches original column means.
result_rescaled = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
    preserve_column_means=True,   # default — recommended
)

# To disable rescaling:
# result_no_rescale = tsam.aggregate(
#     data, n_clusters=6, period_duration="1D",
#     cluster=ClusterConfig(method="hierarchical"),
#     preserve_column_means=False,
# )

# scale_by_column_means (on ClusterConfig) is a *clustering* pre-step:
# it divides each column by its mean before clustering, so all columns
# contribute equally even if they differ in magnitude.
cfg_scaled = ClusterConfig(method="hierarchical", scale_by_column_means=True)
print('scale_by_column_means config:', cfg_scaled)
print('preserve_column_means result RMSE:', round(result_rescaled.accuracy.weighted_rmse, 4))

scale_by_column_means config: ClusterConfig(include_period_sums=False, method='hierarchical', representation=None, scale_by_column_means=True, solver='highs', use_duration_curves=False)
preserve_column_means result RMSE: 0.1235


In [3]:
# Compute the rescaling factor manually on the tiny dataset
# (k=3 Lloyd assignments, mean representation).
numerator_per_attr = normalized.sum(axis=0)
print("Numerator (sum of normalized values per attribute):")
print(numerator_per_attr.round(4))

counts = np.array([(assignments == kk).sum() for kk in range(k)])
print("\nCluster occurrence counts:", counts)

# centers shape (k, 8) attribute-major: first 4 = solar, last 4 = load.
centers_3d = centers.reshape(k, N_ATTRS, N_TIMESTEPS)   # (3, 2, 4)
denom_per_attr = np.sum(counts[:, None] * centers_3d.sum(axis=2), axis=0)
print("\nDenominator (occurrence-weighted centroid sums per attribute):")
print(dict(zip(["solar", "load"], denom_per_attr.round(4))))

rescale_factor = numerator_per_attr.values / denom_per_attr
print("\nRescaling factors per attribute:")
print(dict(zip(["solar", "load"], rescale_factor.round(4))))
print("\n(Factor = 1.0: no correction needed. != 1.0: means drifted during clustering.)")

Numerator (sum of normalized values per attribute):
solar    5.1250
load     7.5714
dtype: float64

Cluster occurrence counts: [2 2 2]

Denominator (occurrence-weighted centroid sums per attribute):
{'solar': np.float64(5.125), 'load': np.float64(7.5714)}

Rescaling factors per attribute:
{'solar': np.float64(1.0), 'load': np.float64(1.0)}

(Factor = 1.0: no correction needed. != 1.0: means drifted during clustering.)


In [4]:
# tsam: with vs without rescaling on real data
result_no_rescale = tsam.aggregate(
    data, n_clusters=6, period_duration="1D",
    cluster=ClusterConfig(method="kmeans"),
    preserve_column_means=False,
)
result_with_rescale = tsam.aggregate(
    data, n_clusters=6, period_duration="1D",
    cluster=ClusterConfig(method="kmeans"),
    preserve_column_means=True,
)

orig_mean = data.mean()
comparison = pd.DataFrame({
    "original_mean": orig_mean,
    "reconstructed_no_rescale": result_no_rescale.reconstructed.mean(),
    "reconstructed_with_rescale": result_with_rescale.reconstructed.mean(),
})
print("Column means — rescaling restores exact match to original:")
comparison.round(4)

C:\Users\j.belina\AppData\Local\miniforge3\envs\tsam_improve_reworked_notebooks\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Column means — rescaling restores exact match to original:


,original_mean,reconstructed_no_rescale,reconstructed_with_rescale
GHI,31.1954,31.1954,31.1954
T,0.4479,0.4479,0.4479
Wind,4.2428,4.2428,4.2428
Load,502.6290,502.6290,502.6290


---

## 2  Denormalisation

After rescaling, representatives are converted back to original units (pipeline
Phase 3):

$$
c'^*_{k,a,t} = c^*_{k,a,t} \left(\max x'_a - \min x'_a\right) + \min x'_a
\quad \forall\; a
$$

This is the exact inverse of the min-max normalisation from
[Preprocessing](01_preprocessing.ipynb).

In [5]:
# Denormalize the cluster 0 centroid back to original units.
# D is attribute-major, so the first 4 coordinates are solar, the last 4 are load.
c0_solar_norm = centroid_0[:N_TIMESTEPS]
c0_load_norm  = centroid_0[N_TIMESTEPS:]

solar_min, solar_max = float(col_min["solar"]), float(col_max["solar"])
load_min,  load_max  = float(col_min["load"]),  float(col_max["load"])

c0_solar_orig = c0_solar_norm * (solar_max - solar_min) + solar_min
c0_load_orig  = c0_load_norm  * (load_max  - load_min)  + load_min

print("Cluster 0 centroid denormalized to original units:")
print(f"  solar (t0..t3): {c0_solar_orig.round(3)}")
print(f"  load  (t0..t3): {c0_load_orig.round(3)}")
print(f"\n  (solar range [{solar_min}, {solar_max}], load range [{load_min}, {load_max}])")

Cluster 0 centroid denormalized to original units:
  solar (t0..t3): [0.  7.5 6.5 0. ]
  load  (t0..t3): [3.  3.  4.  4.5]

  (solar range [0.0, 8.0], load range [3.0, 10.0])


---

**See also:**
* [Extreme periods](08_extreme_periods.ipynb) — forcing peak/extreme periods to survive aggregation
* [Representation strategies](06_representation.ipynb) — the step before rescaling